# Simulator

The data-generation pipeline on its own. **No transformer is imported in this
notebook** - nothing here depends on a checkpoint existing, and every panel is a
property of the physics and the channel alone. Use `transformer.ipynb` for the
model side.

Every parameter is a free-text box rather than a slider, so any value of any
magnitude is accepted. Wrong ones raise rather than clamp.

## Observation channels and bin counts

`bins per channel` is the one to read carefully. Each system exposes more than one
observable, and this box says how many of them are binned and how finely:

- `181` - the single channel every committed run used.
- `181x181` - bins two channels and combines them into **one** token, so the
  vocabulary is the product, `32761`. For the double pendulum that is
  `(theta2, theta1)`: watching `theta2` alone throws away the driven joint, which
  is exactly why that system is the hardest of the four.
- `64x32x16` would work too, for a system exposing three channels.

A product vocabulary is not free. Two correlated channels trace a curve through
the grid rather than filling it, so most cells are never visited - the occupancy
panel reports the fraction, and it drops sharply with each channel added. A larger
vocabulary also means a larger embedding and a higher uniform-prediction loss
(`ln(vocab)`), so a loss number is only comparable within one bin setting.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import ipywidgets as W
import numpy as np
from IPython.display import display

from models.lab import counterfactual, survey_systems, tunable_fields
from models.labplots import plot_all_systems, plot_generation
from physics.messk_configs import CHAIN, DRIVER, MESSK_CONFIGS, SYSTEMS, make_process

STATE = {"proc": None, "batch": None, "survey": None}


def num(label, value, width="230px"):
    """Unbounded numeric entry. No slider caps: any parameter, any magnitude."""
    cls = W.IntText if isinstance(value, (int, np.integer)) and not isinstance(value, bool) else W.FloatText
    return cls(value=value, description=label, style={"description_width": "125px"},
               layout=W.Layout(width=width))

## Controls

In [ ]:
sys_w     = W.Dropdown(options=sorted(MESSK_CONFIGS), value="double_pendulum_mess4",
                       description="system", style={"description_width": "125px"})
delta_v_w = num("delta_v", 1.2)
dt_w      = num("dt", 0.01)
n_steps_w = num("physics steps/tick", 10)
m_w       = num("chain ticks (m)", 16)
bins_w    = W.Text(value="181", description="bins per channel",
                   style={"description_width": "125px"}, layout=W.Layout(width="300px"))
ranges_w  = W.Textarea(value="", description="channel ranges",
                       style={"description_width": "125px"},
                       layout=W.Layout(width="300px", height="52px"))
stay_w    = num("stay", CHAIN["stay"])
alpha_w   = num("alpha", CHAIN["alpha"])
nseq_w    = num("sequences", 256)
seed_w    = num("seed", 20260829)
seq_w     = num("show sequence", 0)
tick_w    = num("branch at tick", 4)

autofit_b = W.Button(description="auto-fit channel 0", button_style="info")
gen_b     = W.Button(description="generate", button_style="primary")
gen_out   = W.Output()
physics_box = W.VBox([])
channel_note = W.HTML()


def _parse_bins():
    parts = [int(p) for p in bins_w.value.replace("x", ",").split(",") if p.strip()]
    return parts[0] if len(parts) == 1 else tuple(parts)


def _rebuild(*_):
    name = MESSK_CONFIGS[sys_w.value]
    system = SYSTEMS[name]["factory"]()
    physics_box.children = [num(f, v) for f, v in tunable_fields(system).items()]
    delta_v_w.value, dt_w.value = SYSTEMS[name]["delta_v"], SYSTEMS[name]["dt"]
    names = getattr(system, "observable_names", ("observable",))
    default = [system.obs_range] + list(getattr(system, "obs_ranges", (system.obs_range,))[1:])
    ranges_w.value = "\n".join(f"{lo:.6g},{hi:.6g}" for lo, hi in default)
    channel_note.value = (f"<b>channels available:</b> {', '.join(names)} &mdash; give one bin "
                          f"count per channel you want binned, e.g. <code>181</code> or "
                          f"<code>181x181</code>")

sys_w.observe(_rebuild, names="value")
_rebuild()


def build_proc():
    overrides = {w.description: w.value for w in physics_box.children}
    rows = [r for r in ranges_w.value.strip().splitlines() if r.strip()]
    parsed = [tuple(float(x) for x in r.split(",")) for r in rows]
    bins = _parse_bins()
    n_channels = 1 if isinstance(bins, int) else len(bins)
    overrides["obs_range"] = parsed[0]
    proc = make_process(
        sys_w.value, system=overrides,
        chain={"stay": stay_w.value, "alpha": alpha_w.value},
        delta_v=delta_v_w.value, dt=dt_w.value, n_steps=int(n_steps_w.value),
        m=int(m_w.value), obs_bins=bins,
    )
    if len(parsed) < n_channels:
        raise ValueError(f"{n_channels} channels binned but {len(parsed)} ranges given")
    return proc


def _generate(_=None):
    with gen_out:
        gen_out.clear_output(wait=True)
        proc = build_proc()
        batch = proc.sample_batch(np.random.default_rng(int(seed_w.value)), int(nseq_w.value))
        STATE["proc"], STATE["batch"] = proc, batch
        r = proc.bin_report(batch["observable"])
        grid = "x".join(map(str, r["obs_bins"]))
        print(f"{int(nseq_w.value)} sequences x {proc.seq_len} tokens | vocabulary "
              f"{grid} = {r['n_obs']} | {r['used_bins']} cells occupied "
              f"({r['used_bins'] / r['n_obs']:.1%} of the grid)")
        for name, clipped, (lo, hi) in zip(r["channels"], r["per_channel_clipped"], proc.obs_ranges):
            values = batch["observable"][..., proc.channel_names.index(name)]
            print(f"  {name:>12s}  range {lo:+.3f}..{hi:+.3f}  observed "
                  f"{values.min():+.3f}..{values.max():+.3f}  clipped {clipped:.2%}")
        display(plot_generation(proc, batch, int(seq_w.value) % len(batch["tokens"]),
                                int(tick_w.value) % proc.m,
                                counterfactual(proc, batch["letters"][int(seq_w.value) % len(batch["tokens"])],
                                               int(tick_w.value) % proc.m)))


def _autofit(_=None):
    proc = build_proc()
    sample = proc.sample_batch(np.random.default_rng(int(seed_w.value)), 128)
    lo, hi = proc.suggested_obs_range(sample["observable"])
    rows = ranges_w.value.strip().splitlines()
    rows[0] = f"{lo:.6g},{hi:.6g}"
    ranges_w.value = "\n".join(rows)
    _generate()

gen_b.on_click(_generate)
autofit_b.on_click(_autofit)

display(W.VBox([
    channel_note,
    W.HBox([
        W.VBox([W.HTML("<b>physics</b>"), sys_w, delta_v_w, dt_w, physics_box]),
        W.VBox([W.HTML("<b>observation channel</b>"), n_steps_w, m_w, bins_w, ranges_w, autofit_b]),
        W.VBox([W.HTML("<b>chain and sampling</b>"), stay_w, alpha_w, nseq_w, seed_w,
                seq_w, tick_w, gen_b]),
    ]),
    gen_out,
]))
_generate()

## All four systems

The same panels for every system under one shared chain and channel setting. The
belief is not drawn per row: the chain runs *in front of* the physics, so
`P(next mood | letters)` is the same object in all four, and what the rows compare
is only the channel.

In [ ]:
survey_bins_w = W.Text(value="181", description="bins per channel",
                       style={"description_width": "125px"}, layout=W.Layout(width="300px"))
survey_n_w    = num("sequences", 256)
survey_tick_w = num("branch at tick", 4)
survey_b      = W.Button(description="survey all four", button_style="primary")
survey_out    = W.Output()

def _survey(_=None):
    with survey_out:
        survey_out.clear_output(wait=True)
        parts = [int(p) for p in survey_bins_w.value.replace("x", ",").split(",") if p.strip()]
        survey = survey_systems(
            rng_seed=int(seed_w.value), n=int(survey_n_w.value), tick=int(survey_tick_w.value),
            m=int(m_w.value), n_steps=int(n_steps_w.value),
            obs_bins=parts[0] if len(parts) == 1 else tuple(parts),
            chain={"stay": stay_w.value, "alpha": alpha_w.value},
        )
        STATE["survey"] = survey
        head = f"{'system':>18s} {'vocab':>8s} {'occupied':>18s} {'clipped':>8s} {'separation':>11s} {'belief H':>9s}"
        print(head); print("-" * len(head))
        for name, e in survey.items():
            b = e["bins"]
            print(f"{name.replace('_mess4', ''):>18s} {b['n_obs']:>8d} "
                  f"{b['used_bins']:>8d} ({b['used_bins'] / b['n_obs']:>5.1%}) "
                  f"{b['clipped']:>7.2%} {e['separation']:>11.1f} {e['belief_entropy']:>9.3f}")
        display(plot_all_systems(survey))

survey_b.on_click(_survey)
display(W.VBox([W.HBox([survey_bins_w, survey_n_w, survey_tick_w, survey_b]), survey_out]))

## Things worth trying

- **`181x181` on the double pendulum.** Both joints become visible in the token.
  Watch the occupancy fraction collapse at the same time - that is the trade.
- **Damping to zero.** The trajectory stops decaying and the occupancy piles up at
  the turning points where the system spends most of its time.
- **`bins per channel` down to `21`.** In the counterfactual panel the four
  alternate futures start landing in the same bin, so the letters stop being
  distinguishable from the observations at all.
- **`delta_v` up until clipping appears.** Clipped samples lose their value, so the
  kick becomes easier to see for the wrong reason.
- **`stay` towards 0.99.** The belief flattens onto one corner and its entropy
  drops, at which point a high downstream R2 stops meaning anything.